### import packages

In [35]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### mock example

In [3]:
prompt = """You are a helpful assistant.
Return an answer to the question
Question: what is your name?"""

In [14]:
response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning_effort="none" 
)

print(response.choices[0].message.content)

My name is **ChatGPT**.


In [5]:
response

ChatCompletion(id='chatcmpl-DxtMI2NHKnMWniIXME2BDg3MRUh7Q', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='My name is **ChatGPT**.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1783166522, model='gpt-5.4-nano-2026-03-17', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=11, prompt_tokens=26, total_tokens=37, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [19]:
response.usage

CompletionUsage(completion_tokens=11, prompt_tokens=26, total_tokens=37, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

### Add Instructor (Structured Outputs)

- why response cost more tokens: The main reason: Pydantic structured output sends the model your schema and makes it output JSON, so both input and output can be larger than a plain text answer.

In [8]:
## completion api vs response api from openai, response api is more recommended from openai
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

class Answer(BaseModel):
    answer: str = Field(description="Answer to the question")

In [9]:
## response api
## response will be a pydantic object, cost a bit more tokens
response = client.create(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [10]:
response

Answer(answer='I’m ChatGPT, an AI assistant.')

In [11]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [12]:
raw_response

Response(id='resp_09984cd2585ca907006a48f7bdc59481a09335dd17a8b9c597', created_at=1783166909.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"I\'m ChatGPT."}', call_id='call_UfoXpPF9bCI09fP0P52p4qkI', name='Answer', type='function_call', id='fc_09984cd2585ca907006a48f7bea8dc81a0b6608ab1a4f73c57', status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', description='Correctly extracted `Answer` with all the required parameters with correct types')], top_p=0.98, background=False, completed_at=1783166910.0, conversation

In [13]:
raw_response.usage

ResponseUsage(input_tokens=82, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=21, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=103)

### rag pipeline

In [ ]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [32]:
import openai
from langsmith import traceable, get_current_run_tree 
# for cost calculation
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(url = "http://localhost:6333") # on docker image it called qdrant as defined

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata['usage_metadata'] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens
        }
    return response.data[0].embedding

def retrieve_from_qdrant(query, k=5):
    response = qdrant_client.query_points(
        collection_name="amazon-electronics-items-collection-01",
        query=get_embedding(query),
        limit=k
    )
    retrieved_context_ids = []
    retrived_context = []
    similarity_scores = []
    retrived_context_ratings = []
    for point in response.points:
        retrieved_context_ids.append(point.payload['parent_asin'])
        retrived_context.append(point.payload['preprocessed_description'])
        similarity_scores.append(point.score)
        retrived_context_ratings.append(point.payload['average_rating'])
        
    return {"retrieved_context_ids": retrieved_context_ids,
            "retrived_context": retrived_context,
            "similarity_scores": similarity_scores,
            "retrived_context_ratings": retrived_context_ratings}

def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrived_context'], context['retrived_context_ratings']):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    prompt = f"""
    You are a helpful shopping assistant that can answer questions about the product in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available product.
    - Do not use markdown formatting.

    Here is the context:
    {preprocessed_context}
    Here is the question:
    {question}
    """
    return prompt


def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )
    # current_run = get_current_run_tree()
    # if current_run:
    #     current_run.metadata['usage_metadata'] = {
    #         "input_tokens": response.usage.prompt_tokens,
    #         "output_tokens": response.usage.completion_tokens,
    #         "total_tokens": response.usage.total_tokens
    #     }
    return response


def rag_pipeline(question, k=5):
    retrieved_context = retrieve_from_qdrant(question, k=k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context['retrieved_context_ids'],
        "retrieved_context": retrieved_context['retrived_context']
    }
    return final_answer


In [33]:
output = rag_pipeline("any usd chargeable speaker")

In [34]:
output

{'data_object': RAGGenerationResponse(answer='The available items are: a 10-ft USB power/charging cable for Ultimate Ears wireless charging docks (cable only), a compact 4-channel amplifier power pack, an Echo Dot wall mount hanger stand (mount/holder only), a set of 3.5mm replacement headphone/mic cables, and a wireless audio receiver (receiver only). There isn’t an actual “USB chargeable speaker” among these items.'),
 'answer': 'The available items are: a 10-ft USB power/charging cable for Ultimate Ears wireless charging docks (cable only), a compact 4-channel amplifier power pack, an Echo Dot wall mount hanger stand (mount/holder only), a set of 3.5mm replacement headphone/mic cables, and a wireless audio receiver (receiver only). There isn’t an actual “USB chargeable speaker” among these items.',
 'question': 'any usd chargeable speaker',
 'retrieved_context_ids': ['B0894DXZFP',
  'B008H7P8GY',
  'B07MJPMQCH',
  'B08DR4K78X',
  'B07FZ3G7SC'],
 'retrieved_context': ['ienza Long Whi